# 🔤 Notebook 03: WordPiece Tokenizer

WordPiece is a subword tokenization algorithm used in models like **BERT**. It improves upon BPE by selecting merges based on maximizing language likelihood rather than just frequency.

### Goals:
- Understand how WordPiece differs from BPE
- Implement a basic WordPiece tokenizer
- Use greedy longest-match for tokenization


In [2]:
# Sample corpus

corpus = [
    "playing",
    "played",
    "player",
    "play",
    "replay"
]

## Step 1: Initialize character-level vocab


In [3]:
from collections import Counter

# Initialize vocabulary by splitting words into characters and appending '<\w>' to mark down end
def build_initial_vocab(corpus) :
    vocab = Counter()
    for word in corpus :
        chars = list(word) + [r'<\w>']  # word = 'play' -> ['p', 'l', 'a', 'y', '<\w>']
        tokens = ' '.join(chars)   # 'p l a y <\w>' -> stored as single token
        vocab[tokens] += 1  # Count frequency
    return vocab

# Create vocab from corpus
vocab = build_initial_vocab(corpus)

# Print initial vocab state
for k,v in vocab.items():
    print(f"{k} : {v}")


p l a y i n g <\w> : 1
p l a y e d <\w> : 1
p l a y e r <\w> : 1
p l a y <\w> : 1
r e p l a y <\w> : 1


## Step 2: Build pair stats like in BPE

In [4]:
# Count all adjacent symbol pairs (bigrams) across all tokens
def get_stats(vocab) :
    pairs = Counter()
    for word, freq in vocab.items() :
        symbols = word.split()  # Split token back into list of symbols
        for i in range(len(symbols) - 1) :
            pairs[(symbols[i], symbols[i+1])] += freq # Count how often each pair occurs
    return pairs


## Step 3: Select best merge (based on frequency here as approximation)

In [5]:
# Merge the most frequent symbol pairs into a new token
def merge_vocab(pair, vocab) :
    new_vocab = {}
    bigram = ' '.join(pair)  # Convert ('p', 'l') -> 'p l' 
    replacement = ''.join(pair) # Convert to 'pl' 
    for word in vocab :
        new_word = word.replace(bigram, replacement) # Replace 'p l' with 'pl' 
        new_vocab[new_word] = vocab[word]
    return new_vocab


## Step 4: Training loop (with limited merges)


In [7]:
#  Perfrom N merge operations on the vocab
def train_wordpiece(corpus, num_merges = 10) :
    vocab = build_initial_vocab(corpus)

    for i in range(num_merges) :
        pairs = get_stats(vocab) # Get all symbol pairs and frequencies
        if not pairs :
            break
        best = max(pairs, key = pairs.get)  # Choose the most frequent pair
        vocab = merge_vocab(best, vocab) # Merge the best pair
        print(f"Step {i+1} :  Merged {best} -> {''.join(best)}")
    return vocab

# Train with 10 merges
trained_vocab = train_wordpiece(corpus, num_merges=10)


Step 1 :  Merged ('p', 'l') -> pl
Step 2 :  Merged ('pl', 'a') -> pla
Step 3 :  Merged ('pla', 'y') -> play
Step 4 :  Merged ('play', 'e') -> playe
Step 5 :  Merged ('play', '<\\w>') -> play<\w>
Step 6 :  Merged ('play', 'i') -> playi
Step 7 :  Merged ('playi', 'n') -> playin
Step 8 :  Merged ('playin', 'g') -> playing
Step 9 :  Merged ('playing', '<\\w>') -> playing<\w>
Step 10 :  Merged ('playe', 'd') -> played


## Step 5: Tokenization using trained vocab (greedy longest match)


In [8]:
# Tokenize input words using trained WordPiece vocabulary
def tokenize_wordpiece(word, vocab) :
    tokens = []
    i = 0
    while i<len(word) :
        # Try to find the longest matching substring in the vocab
        for j in range(len(word), i , -1) :
            sub = word[i:j]
            if i > 0 :
                sub = '##' + sub  # Prefix non-initial subwords with ##
            if sub in vocab :
                tokens.append(sub)
                i = j
                break
        else :
            # If no subword match is found, mark as unknown
            tokens.append('[UNK]')
            break
    return tokens

# Flatten the trianed vocab into a set of subwords
subwords = set()
for word in trained_vocab :
    for token in word.split():
        subwords.add(token.replace(" ", ''))

# Try tokenizing a new word using the trained WordPiece vocab
print(tokenize_wordpiece("replaying", subwords))
    


['r', '[UNK]']


## ✅ Summary

- WordPiece uses a greedy **longest-match-first** tokenizer.
- Improves over BPE by considering word structure more accurately.
- Widely used in Google's models like BERT.
- Future improvement: use likelihood (instead of raw frequency) to select merges.
